> ## SOLUTION / ANSWER KEY &mdash; Lab 4.9
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-09-frozen-vs-finetune.ipynb`](../lab-09-frozen-vs-finetune.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.9 &mdash; Data Efficiency: How Much Training Data Do You Need?

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Train the same head on growing fractions of data
- Plot a learning curve of validation accuracy vs train size
- See why transfer learning shines with little data

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Transfer learning](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = "/tmp/biaa-lab-04-09"
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
A big reason to use pre-trained models: they need **little task data**. We extract **real** frozen
bert-tiny features once, then train the head on growing slices of the training set and watch
validation accuracy rise &mdash; a **learning curve**. With good frozen features, even a few dozen
examples go a long way.

## Build it
For each fraction, train on the first k examples and record validation accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def curve(Xtr, ytr, Xval, yval):
    n = Xtr.shape[0]
    accs = []
    for frac in [0.25, 0.5, 1.0]:
        k = max(2, int(n * frac))
        head = LogisticRegression(max_iter=1000)
        head.fit(Xtr[:k], ytr[:k])
        acc = accuracy_score(yval, head.predict(Xval))
        accs.append(acc)
    return accs

## Run it for real
Extract real features once, then sweep training-set size and plot the curve.

In [ ]:
try:
    import numpy as np
    _FE = {}
    def extract_features(texts):
        """REAL frozen features: forward pass through prajjwal1/bert-tiny, mean-pool over real tokens."""
        import torch
        from transformers import AutoTokenizer, AutoModel
        if not _FE:
            name = "prajjwal1/bert-tiny"
            _FE["tok"] = AutoTokenizer.from_pretrained(name)
            _FE["mdl"] = AutoModel.from_pretrained(name); _FE["mdl"].eval()
        if isinstance(texts, str): texts = [texts]
        enc = _FE["tok"](texts, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad(): out = _FE["mdl"](**enc)          # frozen: no gradients, no weight updates
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)   # mean over REAL tokens only
        return pooled.numpy()
    # A tiny labelled sentiment dataset (1 = positive, 0 = negative)
    SENT = [
        ("i love this it is great", 1), ("a great and wonderful film", 1),
        ("truly wonderful i love it", 1), ("excellent and brilliant work", 1),
        ("the best most brilliant story", 1), ("i love how great it is", 1),
        ("wonderful excellent and great fun", 1), ("a brilliant and great success", 1),
        ("great fun i really love it", 1), ("the best film wonderful and brilliant", 1),
        ("excellent great and lovely work", 1), ("i love this brilliant great film", 1),
        ("wonderful great and the best", 1), ("so good i love it great", 1),
        ("i hate this it is terrible", 0), ("a terrible and awful film", 0),
        ("truly awful i hate it", 0), ("boring and terrible work", 0),
        ("the worst most boring story", 0), ("i hate how bad it is", 0),
        ("awful boring and dull mess", 0), ("a terrible and bad failure", 0),
        ("boring mess i really hate it", 0), ("the worst film awful and boring", 0),
        ("terrible bad and dull work", 0), ("i hate this awful boring film", 0),
        ("awful terrible and the worst", 0), ("so bad i hate it terrible", 0),
    ]
    texts  = [t for t, y in SENT]
    labels = [y for t, y in SENT]
    from sklearn.model_selection import train_test_split
    Xtr_t, Xval_t, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
    Xtr = extract_features(Xtr_t); Xval = extract_features(Xval_t)   # REAL frozen features
    accs = curve(Xtr, ytr, Xval, yval)
    print("val accuracy at 25% / 50% / 100% of train data:", [round(a, 3) for a in accs])
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    plt.plot([0.25, 0.5, 1.0], accs, "o-"); plt.xlabel("train fraction"); plt.ylabel("val accuracy")
    plt.title("Learning curve (frozen real features)"); plt.tight_layout()
    plt.savefig(WORK + "/learning_curve.png", dpi=90)
    print("saved:", WORK + "/learning_curve.png")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- Accuracy generally **climbs with more data**, then plateaus &mdash; the classic learning curve.
- Even the **smallest slice** already does well, because the **frozen features carry meaning**: that is the data-efficiency of transfer learning.
- Curves on tiny data are noisy &mdash; the shape (rise then plateau), not the exact wiggle, is the lesson.

## Your turn (open task &mdash; no grader)
Add more fractions (e.g. `0.1, 0.75`) or more data to `SENT`. Where does the curve
**plateau** &mdash; the point where extra labels stop helping? That plateau is where you would **stop
labelling**. A "good" answer: you can estimate, from the curve, how many labels this task really needs.

---
### Key takeaway
Transfer learning is **data-efficient**: good frozen features mean a small head learns from few labels. Next we stop freezing and fine-tune the whole model.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>